In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

# =========================
# 1. CRIAÇÃO DA BASE  
# =========================

n_linhas = 2500

datas = pd.date_range(start="2025-01-01", end="2025-12-31", freq="D")

regioes = ["Sudeste", "Sul", "Nordeste", "Centro-Oeste", "Norte"]
categorias = ["Tecnologia", "Móveis", "Escritório", "Serviços", "Suprimentos"]

produtos_por_categoria = {
    "Tecnologia": ["Notebook", "Monitor", "Teclado", "Mouse", "Impressora"],
    "Móveis": ["Mesa", "Cadeira", "Armário", "Gaveteiro"],
    "Escritório": ["Papel A4", "Caneta", "Pasta", "Agenda"],
    "Serviços": ["Consultoria", "Suporte Técnico", "Treinamento"],
    "Suprimentos": ["Toner", "Cartucho", "Cabo HDMI", "Filtro de Linha"]
}

canais = ["Online", "Loja Física", "Representante", "Telefone"]
status_pedido = ["Concluído", "Cancelado", "Devolvido"]

clientes = [f"Cliente {i}" for i in range(1, 301)]
vendedores = [f"Vendedor {i}" for i in range(1, 21)]


In [2]:

# Pesos para tornar a base mais realista
pesos_regioes = [0.45, 0.20, 0.15, 0.12, 0.08]
pesos_canais = [0.38, 0.27, 0.25, 0.10]
pesos_status = [0.88, 0.07, 0.05]

linhas = []

for i in range(1, n_linhas + 1):
    data_pedido = np.random.choice(datas)
    regiao = np.random.choice(regioes, p=pesos_regioes)
    categoria = np.random.choice(categorias)
    produto = np.random.choice(produtos_por_categoria[categoria])
    canal = np.random.choice(canais, p=pesos_canais)
    status = np.random.choice(status_pedido, p=pesos_status)
    cliente = np.random.choice(clientes)
    vendedor = np.random.choice(vendedores)
    
    quantidade = np.random.randint(1, 11)
    
    # Preço médio por categoria
    preco_base = {
        "Tecnologia": np.random.uniform(400, 5000),
        "Móveis": np.random.uniform(250, 2500),
        "Escritório": np.random.uniform(10, 200),
        "Serviços": np.random.uniform(500, 8000),
        "Suprimentos": np.random.uniform(50, 800)
    }[categoria]
    
    # Ajuste por região
    fator_regiao = {
        "Sudeste": 1.10,
        "Sul": 1.00,
        "Nordeste": 0.92,
        "Centro-Oeste": 0.95,
        "Norte": 0.88
    }[regiao]
    
    receita_bruta = quantidade * preco_base * fator_regiao
    
    # Cancelados e devolvidos não geram a mesma receita líquida
    if status == "Cancelado":
        receita_liquida = 0
    elif status == "Devolvido":
        receita_liquida = receita_bruta * 0.35
    else:
        receita_liquida = receita_bruta
    
    custo = receita_liquida * np.random.uniform(0.55, 0.82)
    lucro = receita_liquida - custo
    
    linhas.append({
        "id_pedido": i,
        "data_pedido": data_pedido,
        "ano": pd.to_datetime(data_pedido).year,
        "mes": pd.to_datetime(data_pedido).month,
        "ano_mes": pd.to_datetime(data_pedido).strftime("%Y-%m"),
        "cliente": cliente,
        "vendedor": vendedor,
        "regiao": regiao,
        "categoria": categoria,
        "produto": produto,
        "canal": canal,
        "status_pedido": status,
        "quantidade": quantidade,
        "receita_bruta": round(receita_bruta, 2),
        "receita_liquida": round(receita_liquida, 2),
        "custo": round(custo, 2),
        "lucro": round(lucro, 2)
    })

df = pd.DataFrame(linhas)


In [3]:
df.head(3)


,id_pedido,data_pedido,ano,mes,ano_mes,cliente,vendedor,regiao,categoria,produto,canal,status_pedido,quantidade,receita_bruta,receita_liquida,custo,lucro
0,1,2025-04-13,2025,4,2025-04,Cliente 215,Vendedor 11,Nordeste,Escritório,Agenda,Loja Física,Concluído,8,983.80,983.80,790.40,193.41
1,2,2025-07-11,2025,7,2025-07,Cliente 49,Vendedor 10,Norte,Tecnologia,Mouse,Loja Física,Concluído,3,5700.62,5700.62,3828.73,1871.88
2,3,2025-10-01,2025,10,2025-10,Cliente 53,Vendedor 2,Norte,Serviços,Suporte Técnico,Representante,Concluído,4,23756.35,23756.35,15574.34,8182.01


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id_pedido        2500 non-null   int64         
 1   data_pedido      2500 non-null   datetime64[ns]
 2   ano              2500 non-null   int64         
 3   mes              2500 non-null   int64         
 4   ano_mes          2500 non-null   object        
 5   cliente          2500 non-null   object        
 6   vendedor         2500 non-null   object        
 7   regiao           2500 non-null   object        
 8   categoria        2500 non-null   object        
 9   produto          2500 non-null   object        
 10  canal            2500 non-null   object        
 11  status_pedido    2500 non-null   object        
 12  quantidade       2500 non-null   int64         
 13  receita_bruta    2500 non-null   float64       
 14  receita_liquida  2500 non-null   float64

In [5]:

# =========================
# 2. CRIAÇÃO DE MÉTRICAS PARA DASHBOARD
# =========================

df_validos = df[df["status_pedido"] == "Concluído"].copy()

receita_total = df_validos["receita_liquida"].sum()
lucro_total = df_validos["lucro"].sum()
qtd_vendida = df_validos["quantidade"].sum()
total_pedidos = df["id_pedido"].nunique()
pedidos_concluidos = df_validos["id_pedido"].nunique()
clientes_unicos = df_validos["cliente"].nunique()

metricas_gerais = pd.DataFrame({
    "metrica": [
        "Receita total",
        "Lucro total",
        "Margem de lucro",
        "Quantidade vendida",
        "Total de pedidos",
        "Pedidos concluídos",
        "Taxa de conclusão",
        "Ticket médio",
        "Clientes únicos"
    ],
    "valor": [
        receita_total,
        lucro_total,
        lucro_total / receita_total if receita_total > 0 else 0,
        qtd_vendida,
        total_pedidos,
        pedidos_concluidos,
        pedidos_concluidos / total_pedidos if total_pedidos > 0 else 0,
        receita_total / pedidos_concluidos if pedidos_concluidos > 0 else 0,
        clientes_unicos
    ]
})

metricas_gerais["valor"] = metricas_gerais["valor"].round(4)

# Receita mensal para dashboard e análise preditiva
receita_mensal = (
    df_validos
    .groupby(["ano_mes"], as_index=False)
    .agg(
        receita_total=("receita_liquida", "sum"),
        lucro_total=("lucro", "sum"),
        quantidade_vendida=("quantidade", "sum"),
        pedidos=("id_pedido", "nunique")
    )
)

receita_mensal["margem_lucro"] = (receita_mensal["lucro_total"] / receita_mensal["receita_total"]).round(4)
receita_mensal["ticket_medio"] = (receita_mensal["receita_total"] / receita_mensal["pedidos"]).round(2)

# Receita por região
receita_regiao = (
    df_validos
    .groupby("regiao", as_index=False)
    .agg(
        receita_total=("receita_liquida", "sum"),
        lucro_total=("lucro", "sum"),
        quantidade_vendida=("quantidade", "sum"),
        pedidos=("id_pedido", "nunique")
    )
)
receita_regiao["margem_lucro"] = (receita_regiao["lucro_total"] / receita_regiao["receita_total"]).round(4)

# Receita por categoria
receita_categoria = (
    df_validos
    .groupby("categoria", as_index=False)
    .agg(
        receita_total=("receita_liquida", "sum"),
        lucro_total=("lucro", "sum"),
        quantidade_vendida=("quantidade", "sum"),
        pedidos=("id_pedido", "nunique")
    )
)
receita_categoria["margem_lucro"] = (receita_categoria["lucro_total"] / receita_categoria["receita_total"]).round(4)

# Receita por canal
receita_canal = (
    df_validos
    .groupby("canal", as_index=False)
    .agg(
        receita_total=("receita_liquida", "sum"),
        lucro_total=("lucro", "sum"),
        pedidos=("id_pedido", "nunique")
    )
)

# Base mensal para modelo preditivo
base_modelo = receita_mensal.copy()
base_modelo["mes_numero"] = range(1, len(base_modelo) + 1)



In [ ]:
# =========================
# 3. EXPORTAÇÃO DOS ARQUIVOS 
# =========================

output_dir = Path("C:\\Users\\LEGION\\Desktop\\codigos\\puc\\ult sem\\projeto_bi")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / "base_vendas_ficticia.csv", index=False, sep=";", encoding="utf-8-sig")
metricas_gerais.to_csv(output_dir / "metricas_gerais.csv", index=False, sep=";", encoding="utf-8-sig")
receita_mensal.to_csv(output_dir / "receita_mensal.csv", index=False, sep=";", encoding="utf-8-sig")
receita_regiao.to_csv(output_dir / "receita_por_regiao.csv", index=False, sep=";", encoding="utf-8-sig")
receita_categoria.to_csv(output_dir / "receita_por_categoria.csv", index=False, sep=";", encoding="utf-8-sig")
receita_canal.to_csv(output_dir / "receita_por_canal.csv", index=False, sep=";", encoding="utf-8-sig")
base_modelo.to_csv(output_dir / "base_modelo_preditivo.csv", index=False, sep=";", encoding="utf-8-sig")

# Também cria um arquivo compactado com tudo - utilizei auxilio de IA GEN para este bloco.
import zipfile

zip_path = "C:\\Users\\LEGION\\Desktop\\codigos\\puc\\ult sem\\projeto_bi\\base_bi_fase2.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for arquivo in output_dir.glob("*.csv"):
        zipf.write(arquivo, arcname=arquivo.name)

print("Arquivos criados com sucesso:")
for arquivo in sorted(output_dir.glob("*.csv")):
    print(f"- {arquivo}")

print(f"\nArquivo ZIP: {zip_path}")

Arquivos criados com sucesso:
- C:\Users\LEGION\Desktop\codigos\puc\ult sem\projeto_bi\base_modelo_preditivo.csv
- C:\Users\LEGION\Desktop\codigos\puc\ult sem\projeto_bi\base_vendas_ficticia.csv
- C:\Users\LEGION\Desktop\codigos\puc\ult sem\projeto_bi\metricas_gerais.csv
- C:\Users\LEGION\Desktop\codigos\puc\ult sem\projeto_bi\receita_mensal.csv
- C:\Users\LEGION\Desktop\codigos\puc\ult sem\projeto_bi\receita_por_canal.csv
- C:\Users\LEGION\Desktop\codigos\puc\ult sem\projeto_bi\receita_por_categoria.csv
- C:\Users\LEGION\Desktop\codigos\puc\ult sem\projeto_bi\receita_por_regiao.csv

Arquivo ZIP: C:\Users\LEGION\Desktop\codigos\puc\ult sem\projeto_bi\base_bi_fase2.zip
